# 02 - EEG scores: leave-one-subject-out Random Forest in each cohort

**What this notebook does:** trains the primary model of the plan (Random Forest, 11 features, 19 channels) with
leave-one-subject-out (LOSO) inside each cohort. Training is at the epoch level; the subject's P(AD) is the mean over
its epochs. The out-of-fold P(AD) is the EEG score used by notebooks 03, 04 and 06.

**Outputs:** outputs/predictions/oof_{BrainLat,Greek}_RF_11.csv, outputs/results/loso.json.

**Time:** 10-20 min the first time (cached afterwards).

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

from eegad.config import load_context
from eegad.data import load_table
from eegad.reporting import fmt, fmt_ci, fmt_metrics, fmt_p, read_json, write_json

from eegad.models import run_loso
from eegad.stats import auc_ci, threshold_metrics

ctx = load_context("02")
log, out = ctx.log, ctx.out
df_bl, df_gr = load_table(out / "derived" / "brainlat_epochs.csv"), load_table(out / "derived" / "greek_epochs.csv")

In [ ]:
PRED, LOSO = {}, {}
PRED["BrainLat"] = run_loso(df_bl, ctx.features, "RandomForest", "BL_RF_11", ctx)
PRED["Greek"] = run_loso(df_gr, ctx.features, "RandomForest", "GR_RF_11", ctx)
THR = ctx.threshold
for name, pr in PRED.items():
    auc = auc_ci(pr["y"], pr["p"], ctx.n_boot, ctx.seed)
    met = threshold_metrics(pr["y"], pr["p"], THR)
    LOSO[name] = {"AUC": auc, "metrics": met, "n": len(pr)}
    pr.to_csv(out / "predictions" / f"oof_{name}_RF_11.csv", index=False)
    print(f"{name}: EEG AUC (RF, 11 features, 19 channels, LOSO) = {fmt_ci(*auc)} (n = {len(pr)})")
    print(f"   threshold {THR}: {fmt_metrics(met)}")
    log.record("EEG score", f"{name}: EEG AUC (RF, LOSO)", fmt_ci(*auc), "Results; Table 2")
    log.record("EEG score", f"{name}: metrics at threshold {THR} (RF, LOSO)", fmt_metrics(met), "Table 2",
               "descriptive; comparable with the ds004504 studies")
    log.keys_from(f"auc.{name.lower()}.eeg", dict(zip(("value", "lo", "hi"), auc)))
    log.keys_from(f"threshold.{name.lower()}", met)
write_json(out / "results" / "loso.json", LOSO)
print("\nReference on ds004504 (AD vs controls): studies with subject-level validation (Validity 1 of the benchmark by "
      "Miltiadous et al., 2026) report a mean accuracy of 82.1%; with leakage between epochs, about 97%.")